# =============================================================================
# PROJECT: Telecom Customer Churn Prediction
# Author : Mahesh Maddileti
# Course : PGP in Data Science & Machine Learning — IntelliPaat (April 2026)
# Domain : Telecom
# =============================================================================

#
# PROBLEM STATEMENT:
# A telecom company named "Neo" is losing customers to competitors.
# This project analyzes customer data to identify churn patterns and builds
# ML models to predict which customers are at risk of leaving.
#
# DATASET: customer_churn.csv | 7,043 rows | 21 columns
#
# KEY OBJECTIVES:
# 1. Perform data manipulation and exploration
# 2. Visualize patterns in customer behavior
# 3. Build and compare: Linear Regression, Logistic Regression,
#    Decision Tree, and Random Forest models
# =============================================================================

In [ ]:
# ── SECTION 1: IMPORT LIBRARIES ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick


In [ ]:
# ── SECTION 2: LOAD DATASET ──────────────────────────────────────────────────
# Update path below if running locally — replace with your file path
# Example for local: customer_churn = pd.read_csv('customer_churn.csv')

# For Google Colab:
from google.colab import drive
drive.mount('/content/drive')
customer_churn = pd.read_csv("/content/drive/MyDrive/Churn project/customer_churn.csv")

#customer_churn = pd.read_csv('customer_churn.csv')

print("Dataset loaded successfully.")
print(f"Shape: {customer_churn.shape}")
print(f"Columns: {list(customer_churn.columns)}")


In [ ]:
# ── SECTION 3: DATA EXPLORATION ──────────────────────────────────────────────
print("\n--- First 5 rows ---")
print(customer_churn.head())

print("\n--- Last 5 rows ---")
print(customer_churn.tail())

In [ ]:
# ── SECTION 4: DATA MANIPULATION ─────────────────────────────────────────────
# Task 4.1 — Extract the 5th column (index 4)
customer_5 = customer_churn.iloc[:, 4]
print(f"\nColumn 5 (index 4) — Name: {customer_churn.columns[4]}")
print(customer_5.head())

In [ ]:
# Task 4.2 — Extract the 15th column (index 14)
customer_15 = customer_churn.iloc[:, 14]
print(f"\nColumn 15 (index 14) — Name: {customer_churn.columns[14]}")
print(customer_15.head())

In [ ]:
# Task 4.3 — Extract all male senior citizens who pay via Electronic check
# Business use: This segment is a high-risk churn group
senior_male_electronic = customer_churn[
    (customer_churn['gender'] == 'Male') &
    (customer_churn['SeniorCitizen'] == 1) &
    (customer_churn['PaymentMethod'] == 'Electronic check')
]
print(f"\nMale Senior Citizens with Electronic Check payments: {len(senior_male_electronic)} customers")
print(senior_male_electronic.head())

In [ ]:
# Task 4.4 — Extract customers with tenure > 70 months OR monthly charges > $100
# These are long-term or high-value customers
customer_total_tenure = customer_churn[
    (customer_churn['tenure'] > 70) |
    (customer_churn['MonthlyCharges'] > 100)
]
print(f"\nHigh-tenure or high-charge customers: {len(customer_total_tenure)}")
print(customer_total_tenure.head())


In [ ]:
# Task 4.5 — Extract two-year contract + mailed check + churned customers
# Insight: Even long-term customers churn — understanding why matters
two_mail_yes = customer_churn[
    (customer_churn['Contract'] == "Two year") &
    (customer_churn['PaymentMethod'] == "Mailed check") &
    (customer_churn['Churn'] == 'Yes')
]
print(f"\nTwo-year contract + mailed check + churned: {len(two_mail_yes)} customers")
print(two_mail_yes.head())

In [ ]:
# Task 4.6 — Extract 333 random records for sampling analysis
customer_333 = customer_churn.sample(n=333, random_state=42)
print(f"\nRandom sample of 333 customers — Shape: {customer_333.shape}")

In [ ]:
# Task 4.7 — Churn distribution
print("\n--- Churn Value Counts ---")
print(customer_churn['Churn'].value_counts())
# Insight: Class imbalance exists — more non-churners than churners


In [ ]:
# ── SECTION 5: DATA VISUALIZATION ────────────────────────────────────────────
# Plot 5.1 — Distribution of Internet Service types
plt.figure(figsize=(8, 5))
plt.bar(
    customer_churn['InternetService'].value_counts().keys().tolist(),
    customer_churn['InternetService'].value_counts().tolist(),
    color='orange'
)
plt.xlabel('Categories of Internet Service')
plt.ylabel('Count of Categories')
plt.title('Distribution of Internet Service')
plt.tight_layout()
plt.show()
# INSIGHT: Majority of customers use Fiber Optic — this segment also has higher churn

In [ ]:
# Plot 5.2 — Tenure distribution histogram
plt.figure(figsize=(8, 5))
plt.hist(customer_churn['tenure'], bins=30, color='green', edgecolor='white')
plt.xlabel('Months')
plt.ylabel('No. of Customers')
plt.title('Distribution of Customer Tenure')
plt.tight_layout()
plt.show()
# INSIGHT: Large spikes at 0-1 months (new customers) and 70+ months (loyal customers)
# Customers most likely to churn are between 10-65 months


In [ ]:
# Plot 5.3 — Tenure vs Monthly Charges scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(customer_churn['tenure'], customer_churn['MonthlyCharges'], alpha=0.4)
plt.xlabel('Tenure of Customer (months)')
plt.ylabel('Monthly Charges ($)')
plt.title('Tenure vs Monthly Charges')
plt.tight_layout()
plt.show()
# INSIGHT: Newer customers tend to have variable charges; long-tenure customers
# are spread across all charge levels

In [ ]:
# Plot 5.4 — Contract type vs Tenure (box plot)
plt.figure(figsize=(8, 5))
sns.boxplot(x=customer_churn['Contract'], y=customer_churn['tenure'])
plt.xlabel('Contract Type')
plt.ylabel('Tenure (months)')
plt.title('Contract Type vs Customer Tenure')
plt.tight_layout()
plt.show()
# INSIGHT: Two-year contract customers have significantly higher median tenure
# Month-to-month customers leave much sooner — highest churn risk


In [ ]:
# ── SECTION 6: MODEL 1 — LINEAR REGRESSION ───────────────────────────────────
# Goal: Predict MonthlyCharges from tenure
# This helps understand the relationship between how long a customer stays
# and how much they pay

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = np.array(customer_churn['tenure']).reshape(-1, 1)  # Independent variable
y = customer_churn['MonthlyCharges']                    # Dependent variable



In [ ]:
# Split: 70% training, 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred = lr_model.predict(X_test)


In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"\n[Linear Regression] RMSE: {rmse:.2f}")
# Lower RMSE = better prediction accuracy

In [ ]:
# ── SECTION 7: MODEL 2 — LOGISTIC REGRESSION (Simple) ────────────────────────
# Goal: Predict Churn (Yes/No) from MonthlyCharges alone
# Simple model — one independent variable

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

X = np.array(customer_churn['MonthlyCharges']).reshape(-1, 1)
y = customer_churn['Churn']



In [ ]:
# Split: 65% training, 35% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42)

log_model = LogisticRegression()
log_model.fit(X_train, y_train)
y_pred = log_model.predict(X_test)

print("\n[Simple Logistic Regression — MonthlyCharges → Churn]")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# ── SECTION 8: MODEL 3 — LOGISTIC REGRESSION (Multiple) ──────────────────────
# Goal: Predict Churn using both tenure AND MonthlyCharges
# Multiple features = more information = typically better accuracy

X = customer_churn[['tenure', 'MonthlyCharges']]
y = customer_churn['Churn']



In [ ]:
# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

multi_log = LogisticRegression()
multi_log.fit(X_train, y_train)
y_pred = multi_log.predict(X_test)

print("\n[Multiple Logistic Regression — tenure + MonthlyCharges → Churn]")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")


In [ ]:
# ── SECTION 9: MODEL 4 — DECISION TREE CLASSIFIER ────────────────────────────
# Goal: Predict Churn from tenure using a tree-based model
# Decision Trees split data on conditions to classify outcomes

from sklearn.tree import DecisionTreeClassifier

X = np.array(customer_churn['tenure']).reshape(-1, 1)
y = customer_churn['Churn']


In [ ]:
# Split: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)

print("\n[Decision Tree Classifier — tenure → Churn]")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")


In [ ]:
# ── SECTION 10: MODEL 5 — RANDOM FOREST CLASSIFIER ───────────────────────────
# Goal: Predict Churn using tenure + MonthlyCharges
# Random Forest = ensemble of Decision Trees → typically higher accuracy
# Less prone to overfitting than a single Decision Tree

from sklearn.ensemble import RandomForestClassifier

X = customer_churn[['tenure', 'MonthlyCharges']]
y = customer_churn['Churn']



In [ ]:
# Split: 70% training, 30% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("\n[Random Forest Classifier — tenure + MonthlyCharges → Churn]")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# ── SECTION 11: MODEL COMPARISON SUMMARY ─────────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print("Linear Regression       → Predicts MonthlyCharges from tenure")
print("Logistic Regression (S) → Single feature: MonthlyCharges")
print("Logistic Regression (M) → Two features: tenure + MonthlyCharges")
print("Decision Tree           → Single feature: tenure")
print("Random Forest           → Two features: tenure + MonthlyCharges")
print("\nBEST MODEL: Multiple Logistic Regression")
print("REASON: Highest accuracy using two features — tenure and MonthlyCharges")

In [ ]:
# ── SECTION 12: BUSINESS RECOMMENDATION ──────────────────────────────────────
print("\n" + "="*60)
print("BUSINESS RECOMMENDATIONS")
print("="*60)
print("1. Offer better incentives for customers to switch to 2-year contracts")
print("   → Two-year contract customers have significantly lower churn rates")
print("2. Target month-to-month customers in their first 12 months")
print("   → This window has the highest churn probability")
print("3. Review pricing for customers with monthly charges above $65")
print("   → High charges correlate with higher churn likelihood")
print("4. Create special retention offers for senior citizens on electronic check")
print("   → This segment shows elevated churn risk")